# Entrenament i obtenció de model mínim amb un rendiment suficientment bó


In [1]:
import os
import sys
import json
import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt

# Ensure project root is on path
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "model").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from model.model_dylan import FlexibleTNet

print(f"Project root: {PROJECT_ROOT}")
print(f"PyTorch: {torch.__version__}")
device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Project root: /Users/dylancanning/Documents/TFG/Tcav_Captum/Testing-with-Concept-Activation-Vectors
PyTorch: 2.8.0
Device: mps


## Dataset AIXI-Shape

Carreguem les imatges i les etiquetes. La funció `discrete` defineix l'etiqueta binària:

$$y = \mathbb{1}\left[(1 \cdot c) - (0.5 \cdot s) + (0.25 \cdot cr) \geq 0\right]$$

On `c` = cercles, `s` = quadrats, `cr` = creus.

In [2]:
def discrete(c, s, cr):
    """Binary label: ((1*circles) - (0.5*squares) + (0.25*crosses)) >= 0"""
    return float(((1 * c) - (0.5 * s) + (0.25 * cr)) >= 0)


class AIXIShapeDataset(Dataset):
    """AIXI-Shape dataset with discrete binary labels."""

    def __init__(self, img_dir, csv_path, transform=None):
        self.img_dir = Path(img_dir)
        self.transform = transform

        df = pd.read_csv(csv_path, sep=";", index_col=0)
        self.labels = [discrete(row["c"], row["s"], row["cr"]) for _, row in df.iterrows()]

        # Only keep images that exist on disk
        self.image_files = sorted(self.img_dir.glob("*.png"))
        # Truncate labels to match available images
        self.labels = self.labels[: len(self.image_files)]

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img = Image.open(self.image_files[idx]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        label = torch.tensor(self.labels[idx], dtype=torch.float32)
        return img, label


img_transform = transforms.Compose([
    transforms.Resize((128, 128), antialias=True),
    transforms.ToTensor(),
])

train_ds = AIXIShapeDataset(
    PROJECT_ROOT / "data" / "aixi_shape" / "train",
    PROJECT_ROOT / "data" / "dades_train.csv",
    transform=img_transform,
)
val_ds = AIXIShapeDataset(
    PROJECT_ROOT / "data" / "aixi_shape" / "val",
    PROJECT_ROOT / "data" / "dades.csv",
    transform=img_transform,
)

pos = sum(train_ds.labels)
neg = len(train_ds.labels) - pos
print(f"Train: {len(train_ds)} imgs  ({pos:.0f} pos / {neg:.0f} neg)")
print(f"Val:   {len(val_ds)} imgs")

# Preview
img, label = train_ds[0]
print(f"Image shape: {img.shape}, label: {label.item()}")

Train: 50000 imgs  (40838 pos / 9162 neg)
Val:   2000 imgs
Image shape: torch.Size([3, 128, 128]), label: 1.0


## Funcions d'entrenament i avaluació

In [6]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device).unsqueeze(1)
        optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        preds = (torch.sigmoid(out) > 0.5).float()
        correct += (preds == labels).sum().item()
        total += imgs.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device).unsqueeze(1)
        out = model(imgs)
        loss = criterion(out, labels)
        total_loss += loss.item() * imgs.size(0)
        preds = (torch.sigmoid(out) > 0.5).float()
        correct += (preds == labels).sum().item()
        total += imgs.size(0)
    return total_loss / total, correct / total


def train_model(model, train_loader, val_loader, device,
                epochs=12, lr=1e-3, patience=4):
    """Train a model with early stopping. Returns history and best val accuracy."""
    model.to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2, factor=0.5)

    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    best_val_acc = 0.0
    best_state = None
    no_improve = 0

    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        va_loss, va_acc = evaluate(model, val_loader, criterion, device)
        scheduler.step(va_loss)

        history["train_loss"].append(tr_loss)
        history["val_loss"].append(va_loss)
        history["train_acc"].append(tr_acc)
        history["val_acc"].append(va_acc)

        cur_lr = optimizer.param_groups[0]["lr"]
        print(f"  Epoch {epoch:02d}/{epochs} | "
              f"Train loss:{tr_loss:.4f} acc:{tr_acc:.4f} | "
              f"Val loss:{va_loss:.4f} acc:{va_acc:.4f} | lr:{cur_lr:.1e}")

        if va_acc > best_val_acc:
            best_val_acc = va_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1

        if no_improve >= patience:
            print(f"  Early stopping (no improvement for {patience} epochs)")
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    model.to(device)
    return history, best_val_acc


print("Training functions defined.")

Training functions defined.


## Creixement progressiu de FlexNet

Entrenem FlexNet amb arquitectures de complexitat creixent fins a assolir **accuracy >= 98%** en validació.

S'utilitzen **totes les 50k imatges** d'entrenament a resolució completa (128x128). Quan una configuració assoleix el target, s'atura el creixement.

In [20]:
BATCH_SIZE = 512
TARGET_ACC = 0.98

# Progressive architectures: grow depth
# Spatial after pooling: 3conv→16x16, 4conv→8x8, 5conv→4x4
CONFIGS = [
    {"conv_channels": [25, 35, 50],           "fc_layers": [128],           "label": "Conv3-FC1"},
    {"conv_channels": [25, 35, 50, 75],       "fc_layers": [128],           "label": "Conv4-FC1"},
    {"conv_channels": [25, 35, 50, 75],       "fc_layers": [256, 128],      "label": "Conv4-FC2"},
    {"conv_channels": [25, 35, 50, 75, 125],  "fc_layers": [500, 250, 50],  "label": "Conv5-FC3"},
]

for cfg in CONFIGS:
    m = FlexibleTNet(num_channels=3, num_classes=1,
                     conv_channels=cfg["conv_channels"],
                     fc_layers=cfg["fc_layers"], size_img=128)
    n = sum(p.numel() for p in m.parameters())
    print(f"  {cfg['label']:12s} | params: {n:>10,}")

results = []
best_overall_model = None
best_overall_acc = 0.0
best_overall_config = None

  Conv3-FC1    | params:  1,663,287
  Conv4-FC1    | params:    673,262
  Conv4-FC2    | params:  1,320,686
  Conv5-FC3    | params:  1,281,706


In [13]:
# Preload entire dataset into memory (avoids disk I/O during training)
from torch.utils.data import TensorDataset
import time

print("Preloading training images into RAM...")
t0 = time.time()
train_imgs = torch.stack([train_ds[i][0] for i in range(len(train_ds))])
train_lbls = torch.tensor(train_ds.labels, dtype=torch.float32)
print(f"  Train: {train_imgs.shape} loaded in {time.time()-t0:.1f}s")

t0 = time.time()
val_imgs = torch.stack([val_ds[i][0] for i in range(len(val_ds))])
val_lbls = torch.tensor(val_ds.labels, dtype=torch.float32)
print(f"  Val:   {val_imgs.shape} loaded in {time.time()-t0:.1f}s")

print(f"\nAll images preloaded into RAM.")

Preloading training images into RAM...


  Train: torch.Size([50000, 3, 128, 128]) loaded in 21.7s


  Val:   torch.Size([2000, 3, 128, 128]) loaded in 0.7s

Memory DataLoaders ready (batch_size=256)


In [ ]:
import time

device = torch.device("cpu")

train_loader = DataLoader(TensorDataset(train_imgs, train_lbls),
                          batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TensorDataset(val_imgs, val_lbls),
                        batch_size=BATCH_SIZE, shuffle=False)

print(f"Device: {device}")
print(f"Train: {len(train_imgs)} imgs | Val: {len(val_imgs)} imgs | batch_size={BATCH_SIZE}")
print(f"  Train batches/epoch: {len(train_loader)} | Val batches/epoch: {len(val_loader)}")

In [ ]:
for i, cfg in enumerate(CONFIGS):
    label = cfg["label"]
    print(f"\n{'='*60}")
    print(f"Config {i+1}/{len(CONFIGS)}: {label}")
    print(f"  conv={cfg['conv_channels']}, fc={cfg['fc_layers']}")
    print(f"{'='*60}")

    model = FlexibleTNet(
        num_channels=3, num_classes=1,
        conv_channels=cfg["conv_channels"], fc_layers=cfg["fc_layers"],
        dropout_p=0.2, do_sigmoid=False, size_img=128,
    )
    n_params = sum(p.numel() for p in model.parameters())
    print(f"  Parameters: {n_params:,}")

    t0 = time.time()
    history, best_val_acc = train_model(
        model, train_loader, val_loader, device,
        epochs=15, lr=1e-3, patience=4,
    )
    elapsed = time.time() - t0

    results.append({"config": cfg, "best_val_acc": best_val_acc,
                     "history": history, "n_params": n_params})
    print(f"\n>>> {label}: val_acc={best_val_acc:.4f} ({elapsed:.0f}s)")

    if best_val_acc > best_overall_acc:
        best_overall_acc = best_val_acc
        best_overall_config = cfg
        best_overall_model = model

    if best_val_acc >= TARGET_ACC:
        print(f"\nTarget {TARGET_ACC*100:.0f}% reached! Stopping growth.")
        break

print(f"\n{'='*60}")
print(f"BEST CONFIG: {best_overall_config['label']}")
print(f"  conv={best_overall_config['conv_channels']}")
print(f"  fc={best_overall_config['fc_layers']}")
print(f"  Best val accuracy: {best_overall_acc:.4f}")
print(f"{'='*60}")

In [ ]:
# Visualize search results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy comparison
labels = [r["config"]["label"] for r in results]
accs = [r["best_val_acc"] for r in results]
colors = ["green" if a >= TARGET_ACC else "steelblue" for a in accs]
axes[0].barh(labels, accs, color=colors)
axes[0].axvline(x=TARGET_ACC, color="red", linestyle="--", label=f"Target {TARGET_ACC*100:.0f}%")
axes[0].set_xlabel("Best Val Accuracy")
axes[0].set_title("Accuracy per configuration")
axes[0].legend()
for i, v in enumerate(accs):
    axes[0].text(v + 0.002, i, f"{v:.4f}", va="center", fontsize=10)

# Training curves of last (best) config
best_hist = results[-1]["history"]
axes[1].plot(best_hist["train_acc"], label="Train acc", marker="o")
axes[1].plot(best_hist["val_acc"], label="Val acc", marker="s")
axes[1].axhline(y=TARGET_ACC, color="red", linestyle="--", alpha=0.7)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_title(f"Training curves: {results[-1]['config']['label']}")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Training curves of best config
best_hist = results[-1]["history"] if best_overall_acc == results[-1]["best_val_acc"] else \
            [r for r in results if r["best_val_acc"] == best_overall_acc][0]["history"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
epochs_range = range(1, len(best_hist["train_acc"]) + 1)

ax1.plot(epochs_range, best_hist["train_loss"], label="Train", marker="o")
ax1.plot(epochs_range, best_hist["val_loss"], label="Val", marker="s")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss (BCEWithLogits)")
ax1.set_title("Loss curves")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(epochs_range, best_hist["train_acc"], label="Train", marker="o")
ax2.plot(epochs_range, best_hist["val_acc"], label="Val", marker="s")
ax2.axhline(y=TARGET_ACC, color="red", linestyle="--", label=f"Target {TARGET_ACC*100:.0f}%")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy")
ax2.set_title("Accuracy curves")
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_ylim(0.85, 1.01)

plt.suptitle(f"Best model: {best_overall_config['label']} (val acc: {best_overall_acc:.4f})", fontsize=14)
plt.tight_layout()
plt.show()

## Guardar el model

In [ ]:
# Save model weights and config
save_dir = PROJECT_ROOT / "weights"
save_dir.mkdir(exist_ok=True)

conv_str = "conv" + str(len(best_overall_config["conv_channels"]))
fc_str = "fc" + str(len(best_overall_config["fc_layers"]))
save_name = f"flexnet_{conv_str}_{fc_str}_discrete.pt"
save_path = save_dir / save_name

torch.save(best_overall_model.state_dict(), save_path)
print(f"Model saved to: {save_path}")
print(f"  Config: conv={best_overall_config['conv_channels']}, fc={best_overall_config['fc_layers']}")
print(f"  Val accuracy: {best_overall_acc:.4f}")

# Also save full config for reproducibility
config_path = save_dir / save_name.replace(".pt", "_config.json")
with open(config_path, "w") as f:
    json.dump({
        "conv_channels": best_overall_config["conv_channels"],
        "fc_layers": best_overall_config["fc_layers"],
        "num_channels": 3,
        "num_classes": 1,
        "dropout_p": 0.2,
        "do_sigmoid": False,
        "size_img": 128,
        "val_accuracy": best_overall_acc,
        "label_function": "discrete: ((1*c) - (0.5*s) + (0.25*cr)) >= 0",
    }, f, indent=2)
print(f"Config saved to: {config_path}")

## Validació visual del model entrenat

In [ ]:
# Visual sanity check on validation samples
best_overall_model.eval()
num_show = 10

fig, axes = plt.subplots(2, 5, figsize=(16, 7))
axes = axes.flatten()

sample_indices = torch.randperm(len(val_imgs))[:num_show]

for idx, si in enumerate(sample_indices):
    img_tensor = val_imgs[si]
    true_label = val_lbls[si].item()

    with torch.no_grad():
        logit = best_overall_model(img_tensor.unsqueeze(0)).item()
        prob = torch.sigmoid(torch.tensor(logit)).item()
    pred_label = 1 if prob > 0.5 else 0

    # Display image
    img_np = img_tensor.permute(1, 2, 0).numpy()
    axes[idx].imshow(img_np)
    color = "green" if pred_label == true_label else "red"
    axes[idx].set_title(
        f"True: {true_label:.0f} | Pred: {pred_label}\nProb: {prob:.3f}",
        fontsize=9, color=color
    )
    axes[idx].axis("off")

plt.suptitle(f"Validation samples - Green=correct, Red=wrong", fontsize=13)
plt.tight_layout()
plt.show()

# Final summary
print(f"\nModel: {best_overall_config['label']}")
print(f"  conv={best_overall_config['conv_channels']}, fc={best_overall_config['fc_layers']}")
print(f"  Val accuracy: {best_overall_acc:.4f} ({'PASS' if best_overall_acc >= TARGET_ACC else 'FAIL'} target {TARGET_ACC*100:.0f}%)")
print(f"  Saved to: {save_path}")